# Student Performance Predictor

This project is a beginner-level machine learning notebook designed to predict student test scores (average score) based on demographic and background factors. The primary goal is to walk through a complete, simple, and readable end-to-end data science workflow without relying on complex optimizations, advanced pandas chaining, or complicated modeling frameworks. It serves as an accessible introduction to data processing and regression analysis.

In [ ]:
import joblib
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Load Data

In [ ]:
students_df = pd.read_csv("StudentsPerformance.csv")
print(students_df.head())
print(students_df.info())
print(students_df.describe())

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(students_df[['math score', 'reading score', 'writing score']].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

plt.figure(figsize=(8, 6))
sns.histplot(students_df['math score'], bins=20, kde=True)
plt.show()

plt.figure(figsize=(8, 6))
sns.boxplot(x='test preparation course', y='math score', data=students_df)
plt.show()

## Data Cleaning

In [ ]:
missing_values = students_df.isnull().sum()
print(missing_values)

duplicate_count = students_df.duplicated().sum()
print(duplicate_count)

## Feature Engineering

In [ ]:
students_df['average score'] = students_df[['math score', 'reading score', 'writing score']].mean(axis=1)

X = students_df.drop(columns=['math score', 'reading score', 'writing score', 'average score'])
y = students_df['average score'].values

print(X.head())
print(y[:5])


## Train-Test Split

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_features = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print("Training data shape:", X_train.shape)


## Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print(f"Linear Regression - MAE: {lr_mae:.4f} | RMSE: {lr_rmse:.4f} | R2: {lr_r2:.4f}")


## Decision Tree Regressor

In [ ]:
dt_model = DecisionTreeRegressor(max_depth=5, min_samples_split=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

dt_mae = mean_absolute_error(y_test, dt_preds)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_preds))
dt_r2 = r2_score(y_test, dt_preds)

print(f"Decision Tree - MAE: {dt_mae:.4f} | RMSE: {dt_rmse:.4f} | R2: {dt_r2:.4f}")


## Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_split=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print(f"Random Forest - MAE: {rf_mae:.4f} | RMSE: {rf_rmse:.4f} | R2: {rf_r2:.4f}")


## Evaluate and Compare

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'MAE': [lr_mae, dt_mae, rf_mae],
    'RMSE': [lr_rmse, dt_rmse, rf_rmse],
    'R2': [lr_r2, dt_r2, rf_r2]
})

print(results_df.sort_values(by='R2', ascending=False))
